# Polymorphism, inheritance and exceptions

**Advanced Computational Physics** &middot; Master in Physics &middot; USC

*J. A. Hernando Morata*. Original material (2020) in collaboration with
*G. Martínez-Lema* and *M. Kekic*.

**Objectives**

* see why classes matter: **polymorphism**, the same operation on different things;
* derive a class from another one: inherit, override, `super()`;
* define an **interface** with an abstract base class;
* decide between **inheritance** ("is a") and **composition** ("has a");
* understand duck typing and the method resolution order;
* use exceptions properly: they are a class hierarchy too.

In [ ]:
import math
from abc import ABC, abstractmethod

## 1. Polymorphism: the same operation on different things

Functions alone are enough to compute anything. So why does almost every serious
scientific code (ROOT, Geant4, astropy, scikit-learn, numpy itself) hand you **objects**?
The main reason shows up when a program must do the **same operation on different
kinds of things**.

A particle moves under a force, and the force can come from several models. Written with
functions, the code has to know about every model:

In [ ]:
def force_of(kind, x, v):
    """Force of the model called kind on a particle at x with velocity v."""
    if kind == 'spring':
        return -1. * x
    elif kind == 'gravity':
        return -9.8
    elif kind == 'drag':
        return -0.5 * v
    else:
        raise ValueError(f'unknown force model {kind!r}')


def simulate_kind(kind, x, v, m, dt, n):
    """Position after n steps of size dt (semi-implicit Euler)."""
    for _ in range(n):
        v += force_of(kind, x, v) / m * dt
        x += v * dt
    return x


print([round(force_of(k, 2., 1.), 2) for k in ('spring', 'gravity', 'drag')])

Every new model means editing `force_of`.

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — How many places?</b></p>

A fourth force model arrives: a constant push. In a real code the `if/elif` chain of
`force_of` appears in the simulation, in the plotting and in the output. **Decide**: how
many places must you edit in this version? And how many existing lines will change in
the version with objects, that comes next?

</div>

<details>
<summary><b>Solution</b></summary>

With `if/elif`: one new branch in **every** chain on the kind of model (simulation,
plotting, output…), and forgetting one of them is a silent bug, or a `ValueError` at run
time. With objects: **none**. You write one new class, and no existing line changes.

</details>

### 1.1 The same thing with objects

Each model is an object that knows how to compute its own force:

In [ ]:
class Spring:
    def __init__(self, k):
        self.k = k

    def force(self, x, v):
        return -self.k * x


class Gravity:
    def __init__(self, g, m):
        self.g, self.m = g, m          # m: the mass of the particle

    def force(self, x, v):
        return -self.m * self.g


class Drag:
    def __init__(self, b):
        self.b = b

    def force(self, x, v):
        return -self.b * v


for model in (Spring(1.), Gravity(9.8, 1.), Drag(0.5)):
    print(f'{type(model).__name__:8s} -> {model.force(2., 1.):6.2f}')

The code that **uses** the models never mentions any of them:

In [ ]:
def simulate(model, x, v, m, dt, n):
    """Position after n steps of size dt (semi-implicit Euler)."""
    for _ in range(n):
        v += model.force(x, v) / m * dt
        x += v * dt
    return x

print(f'spring: {simulate(Spring(1.), 1., 0., 1., 0.01, 100):.4f}')
print(f'drag  : {simulate(Drag(0.5), 0., 1., 1., 0.01, 100):.4f}')

Adding a fourth model requires **no change at all** to `simulate`: the new model only has
to have a method `force(x, v)`.

> **The code that uses a thing does not need to know what the thing is, only what it can
> do.**

That is **polymorphism** (many forms, one interface), and it is why every simulation
framework you will meet is built this way. The *interface* is a promise: "any object with
a `force(x, v)` method can be used by `simulate`".

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop — A fourth force</b></p>

Write a fourth force model, a constant push `force(x, v) = F0`, and use it with
`simulate`. How many lines of existing code did you change? Now add it to the `force_of`
version too. Check that both versions give the same position after 100 steps.

</div>

In [ ]:
# your code here

<details>
<summary><b>Solution</b></summary>

```python
class Push:
    def __init__(self, f0):
        self.f0 = f0

    def force(self, x, v):
        return self.f0

x_obj = simulate(Push(2.), 0., 0., 1., 0.01, 100)

# the procedural version: a new branch, inside force_of
def force_of(kind, x, v):
    if kind == 'spring':
        return -1. * x
    elif kind == 'gravity':
        return -9.8
    elif kind == 'drag':
        return -0.5 * v
    elif kind == 'push':
        return 2.                      # and where does F0 go? another argument...
    else:
        raise ValueError(f'unknown force model {kind!r}')

assert math.isclose(simulate_kind('push', 0., 0., 1., 0.01, 100), x_obj)
```

The object version adds a class and touches nothing. The procedural one edits an existing
function, and has nowhere to put the parameter `F0`: the spring constant and the drag
coefficient are already hard-coded. Objects carry their **parameters** with them.

</details>

<div class="admonition note">
<p class="admonition-title"><b>[!] Discuss — Is a function enough?</b></p>

`simulate` could take a plain function instead, `simulate(lambda x, v: -x, ...)`, and
call it. Then why classes for the force models? Think of a model with several parameters,
one that keeps a state (a random generator, a table read from a file), or one that must
offer several methods (`force`, `potential`, `describe`).

</div>

<details>
<summary><b>Solution</b></summary>

For a single, stateless formula, a function (or a `functools.partial`) is enough, and
simpler: functions are objects too, and their interface is "can be called". A class wins
when the model carries **parameters and state** that belong together, or when the code
needs **several** operations from the same model: `force` and `potential` must use the
same `k`. We come back to this choice in
[Functional or object-oriented?](./fp_or_oop.ipynb).

</details>

Nothing checks the promise: a model without `force` fails only when `simulate` calls it.
**Inheritance** lets us write the promise down, and share code between the classes that
keep it.

## 2. Inheritance

A class can be built on another one. The **derived** (child) class **inherits** all the
attributes and methods of the **base** (parent) class, and can **add** new ones or
**override** (redefine) the inherited ones.

<img src="./img/classes_inheritance.jpeg" width="420">

A car *is a* wheeled vehicle, which *is a* vehicle: everything that can be done with a
vehicle can be done with a car.

### 2.1 An interface: the shapes

A **base class** can define what its derived classes must be able to do, without saying
how. With `ABC` and `@abstractmethod`, Python enforces it: a class with unimplemented
abstract methods cannot be instantiated. (The force models of section 1 could derive
from a `ForceModel(ABC)` with an abstract `force`.)

In [ ]:
class Shape(ABC):
    """Interface of the 2D shapes."""

    @abstractmethod
    def area(self):
        """Return the area."""

    @abstractmethod
    def perimeter(self):
        """Return the perimeter."""

    def describe(self):
        """A concrete method, written with the abstract ones."""
        return f'{type(self).__name__}: area {self.area():.3f}, perimeter {self.perimeter():.3f}'

`describe` is **concrete**: it works for every shape, present or future, because it only
uses the interface.

In [ ]:
class Disk(Shape):

    def __init__(self, radius):
        self.radius = radius

    def area(self):
        return math.pi * self.radius ** 2

    def perimeter(self):
        return 2 * math.pi * self.radius


class Rectangle(Shape):

    def __init__(self, width, height):
        self.width, self.height = width, height

    def area(self):
        return self.width * self.height

    def perimeter(self):
        return 2 * (self.width + self.height)

In [ ]:
for shape in [Disk(1.), Rectangle(2., 3.)]:
    print(shape.describe())

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — Abstract classes</b></p>

What happens with each of these? **When** is the problem reported?

```python
Shape()

class Triangle(Shape):
    def __init__(self, a, b, c):
        self.a, self.b, self.c = a, b, c
    def perimeter(self):
        return self.a + self.b + self.c

Triangle(3., 4., 5.)
```

</div>

In [ ]:
class Triangle(Shape):
    def __init__(self, a, b, c):
        self.a, self.b, self.c = a, b, c
    def perimeter(self):
        return self.a + self.b + self.c

Triangle(3., 4., 5.)

<details>
<summary><b>Solution</b></summary>

Both fail with `TypeError: Can't instantiate abstract class ...` (the cell below only
runs the second one). `Shape` is abstract,
and `Triangle` forgot `area`. The error appears when the object is **created**, with a
clear message. Without `ABC`, a forgotten method would appear much later, in the middle of
a computation, or not at all: a base `area` that returns `None` would make
`sum(s.area() for s in shapes)` fail far from the cause.

</details>

### 2.2 A class derived from a derived class: super()

A square *is a* rectangle with equal sides. It only needs its own constructor, which
calls the constructor of the base class with `super()`:

In [ ]:
class Square(Rectangle):

    def __init__(self, side):
        super().__init__(side, side)

sq = Square(2.)
print(sq.describe())
print(isinstance(sq, Square), isinstance(sq, Rectangle), isinstance(sq, Shape))

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — Which method runs?</b></p>

`Square` defines neither `area` nor `describe`. **Decide** in which class Python finds
each method when we call `sq.describe()`, and in what order it looks.

</div>

In [ ]:
print(Square.__mro__)

<details>
<summary><b>Solution</b></summary>

Python looks for a name in the **method resolution order**, `Square.__mro__`: `Square`,
`Rectangle`, `Shape`, `ABC`, `object`. `describe` is found in `Shape`; inside it,
`self.area()` is looked up again starting from `Square`, and found in `Rectangle`.
`type(self).__name__` is `'Square'`: `self` is always the real object.

</details>

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop — The triangle, done right</b></p>

Complete `Triangle(a, b, c)`: the area with Heron's formula,
$A = \sqrt{s(s-a)(s-b)(s-c)}$ with $s$ the semi-perimeter. The constructor must refuse
sides that cannot form a triangle (use `raise ValueError(...)`; exceptions are covered in
section 6). Test it with the triangle $3, 4, 5$.

</div>

In [ ]:
# your code here

<details>
<summary><b>Solution</b></summary>

```python
class Triangle(Shape):

    def __init__(self, a, b, c):
        if a + b <= c or a + c <= b or b + c <= a:
            raise ValueError(f'{a}, {b}, {c} do not make a triangle')
        self.a, self.b, self.c = a, b, c

    def perimeter(self):
        return self.a + self.b + self.c

    def area(self):
        s = self.perimeter() / 2
        return math.sqrt(s * (s - self.a) * (s - self.b) * (s - self.c))

assert math.isclose(Triangle(3., 4., 5.).area(), 6.)
print(Triangle(3., 4., 5.).describe())
```

</details>

## 3. Duck typing

Code written for the interface works for every shape:

In [ ]:
def total_area(shapes):
    return sum(shape.area() for shape in shapes)

print(total_area([Disk(1.), Rectangle(2., 3.), Square(2.)]))

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — Without the base class</b></p>

This class does **not** derive from `Shape`. **Decide**: does
`total_area([Disk(1.), Ring(1., 2.)])` work?

```python
class Ring:
    def __init__(self, r1, r2):
        self.r1, self.r2 = r1, r2
    def area(self):
        return math.pi * (self.r2 ** 2 - self.r1 ** 2)
```

</div>

In [ ]:
class Ring:
    def __init__(self, r1, r2):
        self.r1, self.r2 = r1, r2
    def area(self):
        return math.pi * (self.r2 ** 2 - self.r1 ** 2)

print(total_area([Disk(1.), Ring(1., 2.)]))

<details>
<summary><b>Solution</b></summary>

It works. `total_area` only calls `area()`, and `Ring` has it: Python does not check the
class, only that the method exists (**duck typing**: if it quacks like a duck…). But
`Ring(1., 2.).describe()` fails with `AttributeError`: `describe` lives in `Shape`, and
`Ring` does not inherit it.

</details>

<div class="admonition note">
<p class="admonition-title"><b>[!] Discuss — Then why inherit?</b></p>

If duck typing is enough, why write `Shape` at all? Think of: documentation, errors at
creation time, shared concrete methods like `describe`, `isinstance` checks. In Java,
and for virtual calls in C++, run-time polymorphism requires inheritance (or an
interface); in Python it does not. When would you skip it?

</div>

<details>
<summary><b>Solution</b></summary>

The base class buys four things: the interface is **written down** (documentation that
cannot go stale); a missing method is reported when the object is **created**, not deep in
a computation; concrete methods like `describe` are written **once**; and
`isinstance(x, Shape)` can check it. Skip it for small, local code, when the "interface"
is one method used in one place (`simulate` and its force models), or when the objects
come from somebody else's code and you cannot change their base class.
`typing.Protocol` is a middle way: it describes an interface for the type checkers
without inheritance.

</details>

## 4. Extending a built-in

We can derive from built-in classes too. A list that can give its descending order:

In [ ]:
class MyList(list):

    def descending(self):
        return sorted(self, reverse=True)

xs = MyList([3, 1, 2])
print(xs, xs.descending(), len(xs), type(xs))

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — What type?</b></p>

What is the type of `xs + xs`, and of `xs[:2]`? Does `(xs + xs).descending()` work?

</div>

In [ ]:
print(type(xs + xs), type(xs[:2]))

<details>
<summary><b>Solution</b></summary>

Both are plain `list`, so `(xs + xs).descending()` fails. The methods of a built-in
**hard-code** the built-in type: they create plain `list` objects, not `MyList`. To keep
the type you must override them all, which is long and fragile (`collections.UserList`
helps). This is one reason to prefer composition (next
section), or a plain function `descending(xs)`.

</details>

## 5. Inheritance or composition?

* **Inheritance** says "**is a**": a `Square` is a `Rectangle`.
* **Composition** says "**has a**": a `Detector` has a `Geometry` and a list of `Channel`s,
  stored as attributes.

Composition is more flexible: the parts can be changed independently. The usual advice:
**prefer composition**, and use inheritance for real "is a" relations, one or two levels
deep. A deep hierarchy (`Detector` → `GaseousDetector` → `TPC` → `HighPressureTPC` → …)
is very hard to change later: every class depends on all its ancestors.

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — Is a, or has a?</b></p>

For each pair, **decide**: inheritance, composition, or neither?

1. `Muon` and `Particle`
2. `Event` and `Particle`
3. `Calorimeter` and `Detector`
4. `Track` and `Hit`
5. `Histogram2D` and `Histogram1D`

</div>

<details>
<summary><b>Solution</b></summary>

1. Inheritance, if at all: a muon *is a* particle. (Often a `Particle` with a `name`
   attribute is enough: one class, many data.)
2. Composition: an event *has* particles.
3. Inheritance: a calorimeter *is a* detector, with the same interface (`read`,
   `calibrate`…).
4. Composition: a track *has* hits.
5. Neither: a 2D histogram is not a 1D histogram (what would `fill(x)` mean?), and it
   does not contain one. Similar names do not make an "is a".

</details>

<div class="admonition note">
<p class="admonition-title"><b>[!] Discuss — Is a square a rectangle?</b></p>

Suppose `Rectangle` gets a method `set_width(w)` that changes the width only. What happens
when it is called on a `Square`? Is `Square` still a valid substitute for a `Rectangle`?
What does this say about "is a" in code versus in geometry?

</div>

<details>
<summary><b>Solution</b></summary>

`Square(2.).set_width(3.)` gives a "square" of $3 \times 2$: the derived class breaks its
own invariant. In geometry a square is a rectangle; in code, *a mutable square is not a
mutable rectangle*, because it cannot do everything a rectangle can do (change one side).
The rule (Liskov substitution principle): a derived object must work **everywhere** the
base object works, and keep its guarantees (after `set_width(w)`, `area == w * height`).
Our shapes only avoid the problem by convention (nothing stops `sq.width = 3.`); truly
immutable objects, or read-only properties, avoid it.

</details>

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop — Has a: an event with particles</b></p>

Composition in code. Write a frozen dataclass (`from dataclasses import dataclass`)
`Particle(name, E, px, py, pz)` and a class
`Event` that **has** a list of particles, with:

* `total_momentum()`, the sum of the three-momenta;
* `invariant_mass()`, the mass of the whole event;
* `select(condition)`, which takes a **function** `condition(particle) -> bool` and returns
  a **new** `Event` with the particles that satisfy it.

Test it with an event of two back-to-back photons of 45 GeV: total momentum 0, mass 90 GeV.
Note that `select` is functional programming inside a class.

</div>

In [ ]:
# your code here

<details>
<summary><b>Solution</b></summary>

```python
from dataclasses import dataclass

@dataclass(frozen=True)
class Particle:
    name: str
    E: float
    px: float
    py: float
    pz: float

class Event:
    """A collision event: it has a list of particles."""

    def __init__(self, particles):
        self.particles = list(particles)

    def total_momentum(self):
        return (sum(p.px for p in self.particles),
                sum(p.py for p in self.particles),
                sum(p.pz for p in self.particles))

    def invariant_mass(self):
        energy = sum(p.E for p in self.particles)
        px, py, pz = self.total_momentum()
        return math.sqrt(max(energy ** 2 - px ** 2 - py ** 2 - pz ** 2, 0.))

    def select(self, condition):
        return Event(p for p in self.particles if condition(p))

ev = Event([Particle('gamma', 45., 0., 0., 45.), Particle('gamma', 45., 0., 0., -45.),
            Particle('pi+', 2., 0.5, 0., 1.)])
photons = ev.select(lambda p: p.name == 'gamma')
assert photons.total_momentum() == (0., 0., 0.)
assert math.isclose(photons.invariant_mass(), 90.)
```

`Event` is not a list and does not derive from `list`: it **has** one. It can change how it
stores the particles (a numpy array, for instance) without changing its interface.

</details>

## 6. Exceptions

### 6.1 Exceptions are classes

Errors are objects of exception classes, organised in a hierarchy. `ZeroDivisionError`
*is an* `ArithmeticError`, which *is an* `Exception`:

In [ ]:
print(ZeroDivisionError.__mro__)

`KeyboardInterrupt` (Ctrl-C) derives from `BaseException`, not from `Exception`:
`except Exception` does not catch it.

The full tree is in the
[documentation](https://docs.python.org/3/library/exceptions.html#exception-hierarchy).
An `except` clause catches the class it names **and all its derived classes**:

In [ ]:
try:
    1. / 0.
except ArithmeticError as e:
    print('caught:', type(e).__name__, '-', e)

The full form: `try` runs the code; the **first** `except E` that matches the error runs;
`else` runs if there was **no** error; `finally` runs **always** (to close files, release
resources).

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — Which except?</b></p>

**Decide** what this prints.

```python
try:
    math.sqrt(-1.)
except ZeroDivisionError:
    print('A')
except ValueError:
    print('B')
except Exception:
    print('C')
else:
    print('D')
finally:
    print('E')
```

</div>

In [ ]:
try:
    math.sqrt(-1.)
except ZeroDivisionError:
    print('A')
except ValueError:
    print('B')
except Exception:
    print('C')
else:
    print('D')
finally:
    print('E')

<details>
<summary><b>Solution</b></summary>

`B` and `E`. `math.sqrt(-1.)` raises `ValueError`; the **first** matching `except` runs,
and only that one; `else` is skipped; `finally` runs. Order matters: put the derived
classes before their base; `except Exception` first would shadow the `ValueError`.

</details>

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — A silent handler</b></p>

This code is meant to handle a zero denominator. What does it print? Is that what you
expected?

```python
x = 2.
try:
    result = 1. / x
    print(reslt)
except:
    print('x is zero')
```

</div>

In [ ]:
x = 2.
try:
    result = 1. / x
    print(reslt)
except:
    print('x is zero')

<details>
<summary><b>Solution</b></summary>

It prints `x is zero`, but `x` is 2. The real error is a typo, `reslt`, a `NameError`,
and the bare `except:` catches **everything** and hides it (it would even catch a
`Ctrl-C`). Always name the exceptions you expect, `except ZeroDivisionError:`, and keep
the `try` block as short as possible.

</details>

### 6.2 Your own exceptions

Derive from the closest built-in exception, so that users can catch it either by its own
name or by its base class:

In [ ]:
class FitError(RuntimeError):
    """The fit failed."""

def fit(data):
    if len(data) < 3:
        raise FitError(f'need at least 3 points, got {len(data)}')
    return sum(data) / len(data)

fit([1., 2.])

In [ ]:
try:
    fit([1., 2.])
except RuntimeError as e:
    print('caught a', type(e).__name__, ':', e)

Raise early, with a message that says what went wrong and with which value. Catch only
where you can **do** something about it; otherwise let it go up.

## 7. Practice

Inheritance on real code: a class from the repository, a contract that several shapes
must honour, and a family of probability distributions.

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop — Vector3D from Vector</b></p>

The file [nvector.py](./nvector.py) has a class `Vector` of $n$ components, with `+`,
multiplication by a number, indexing and `==`.

1. Read it (`from nvector import Vector`). Find at least one bug in `__eq__`: try
   `Vector([1, 2]) == Vector([1, 2, 3])`.
2. Write `Vector3D(Vector)` with a constructor `Vector3D(x, y, z)` and a method `cross`.
3. What is the type of `Vector3D(1, 0, 0) + Vector3D(0, 1, 0)`? Why? Can `cross` be
   applied to it?

</div>

In [ ]:
# your code here

<details>
<summary><b>Solution</b></summary>

```python
from nvector import Vector

print(Vector([1, 2]) == Vector([1, 2, 3]))      # True! zip stops at the shortest

class Vector3D(Vector):
    """A vector of three components."""

    def __init__(self, x, y, z):
        super().__init__([x, y, z])

    def cross(self, other):
        a, b = self.values, other.values
        return Vector3D(a[1] * b[2] - a[2] * b[1],
                        a[2] * b[0] - a[0] * b[2],
                        a[0] * b[1] - a[1] * b[0])

ex, ey = Vector3D(1, 0, 0), Vector3D(0, 1, 0)
print(ex.cross(ey))
print(type(ex + ey))                             # Vector, not Vector3D
```

`__eq__` must first compare the lengths. And `Vector.__add__` builds a `Vector`, so the
sum loses `cross`, as `MyList` lost `descending`. `type(self)(...)` does not help either:
the constructor of `Vector3D` takes three numbers, not a list. Either override the
operators in `Vector3D`, or make all constructors take the same arguments.

</details>

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop (in pairs) — One test suite, many shapes</b></p>

A test of the **contract** of `Shape` works for any shape:

```python
def check_shape(shape):
    assert shape.area() > 0 and shape.perimeter() > 0
    assert isinstance(shape.describe(), str)
```

and, for shapes that can be scaled, `scaled(2)` multiplies the perimeter by 2 and the area
by 4. In pairs: each pair implements a different shape (regular polygon, ring derived
from `Shape`, ellipse with Ramanujan's approximation for the perimeter…) with a method
`scaled(k)`, and writes **one** test that it applies
to the shapes of **all** the pairs. Do all the shapes pass all the tests?

</div>

In [ ]:
# your code here

<details>
<summary><b>Solution</b></summary>

```python
def check_shape(shape):
    assert shape.area() > 0 and shape.perimeter() > 0
    assert isinstance(shape.describe(), str)

class RegularPolygon(Shape):

    def __init__(self, n, side):
        self.n, self.side = n, side

    def perimeter(self):
        return self.n * self.side

    def area(self):
        return self.n * self.side ** 2 / (4 * math.tan(math.pi / self.n))

    def scaled(self, k):
        return RegularPolygon(self.n, k * self.side)

def check_scaling(shape):
    big = shape.scaled(2.)
    assert math.isclose(big.perimeter(), 2 * shape.perimeter())
    assert math.isclose(big.area(), 4 * shape.area())

for shape in [RegularPolygon(6, 1.), RegularPolygon(3, 2.)]:
    check_shape(shape)
    check_scaling(shape)
```

One test, many implementations: this is polymorphism used for testing. In pytest it is a
`@pytest.mark.parametrize` over the shapes.

</details>

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop (in pairs) — A family of distributions</b></p>

A base class `Distribution(ABC)` with an abstract `pdf(x)`, a range `(xmin, xmax)`, and
two **concrete** methods written only with `pdf`: `norm()` (the integral of the pdf, which
must be 1) and `mean()`, both by numerical integration.

In a group: one member writes the base class; the others write `Uniform(a, b)`,
`Exponential(tau)` and `Gaussian(mu, sigma)`, each with a test comparing `norm()` and
`mean()` with the exact values. Then put it all together: does every test pass with
every class?

</div>

In [ ]:
# your code here

<details>
<summary><b>Solution</b></summary>

```python
class Distribution(ABC):

    def __init__(self, xmin, xmax):
        self.xmin, self.xmax = xmin, xmax

    @abstractmethod
    def pdf(self, x):
        """Probability density at x."""

    def _integrate(self, f, n=10_000):
        h = (self.xmax - self.xmin) / n
        return h * sum(f(self.xmin + (i + 0.5) * h) for i in range(n))

    def norm(self):
        return self._integrate(self.pdf)

    def mean(self):
        return self._integrate(lambda x: x * self.pdf(x))


class Exponential(Distribution):

    def __init__(self, tau):
        super().__init__(0., 30. * tau)
        self.tau = tau

    def pdf(self, x):
        return math.exp(-x / self.tau) / self.tau


class Gaussian(Distribution):

    def __init__(self, mu, sigma):
        super().__init__(mu - 10. * sigma, mu + 10. * sigma)
        self.mu, self.sigma = mu, sigma

    def pdf(self, x):
        z = (x - self.mu) / self.sigma
        return math.exp(-0.5 * z * z) / (math.sqrt(2. * math.pi) * self.sigma)

class Uniform(Distribution):

    def __init__(self, a, b):
        super().__init__(a, b)

    def pdf(self, x):
        return 1. / (self.xmax - self.xmin)

for d, mean in [(Uniform(-1., 3.), 1.), (Exponential(2.), 2.),
                (Gaussian(1., 0.5), 1.), (Gaussian(0., 1.), 0.)]:
    assert math.isclose(d.norm(), 1., rel_tol=1e-5)
    assert math.isclose(d.mean(), mean, rel_tol=1e-5, abs_tol=1e-9)   # abs_tol: mean 0
```

The range is a choice of each derived class (an exponential has no upper limit: we cut
at $30\tau$), and the integration lives once, in the base class. This is the same pattern as
`Shape.describe`, and the one that `scipy.stats` follows.

</details>

## 8. Review

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop (review) — A forgotten super()</b></p>

What is wrong, and when does it show up?

```python
class Detector:
    def __init__(self, name):
        self.name = name
        self.hits = []

    def add_hit(self, hit):
        self.hits.append(hit)

class Calorimeter(Detector):
    def __init__(self, name, cells):
        self.cells = cells

calo = Calorimeter('ECAL', 1024)
calo.add_hit(3.2)
```

</div>

<details>
<summary><b>Solution</b></summary>

`Calorimeter.__init__` overrides the base constructor and never calls it, so `name` and
`hits` are never created. The class is defined, the object is created, and the error
appears only later, in `add_hit`: `AttributeError: 'Calorimeter' object has no attribute
'hits'`. Fix:

```python
class Detector:
    def __init__(self, name):
        self.name = name
        self.hits = []

    def add_hit(self, hit):
        self.hits.append(hit)

class Calorimeter(Detector):
    def __init__(self, name, cells):
        super().__init__(name)
        self.cells = cells
```

</details>

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop (review) — Catching too much</b></p>

```python
def fit_all(histograms):
    results = []
    for h in histograms:
        try:
            results.append(fit(h))
        except Exception:
            pass
    return results
```

What does this function hide? How would you rewrite it? (`fit` and `FitError` as in
section 6.2.)

</div>

<details>
<summary><b>Solution</b></summary>

It hides **every** failure, including bugs (a typo, a wrong index) and fits that did not
converge, and it returns fewer results than histograms without saying which ones are
missing. Catch only the error you expect, and keep a record:

```python
def fit_all(histograms):
    results, failed = [], []
    for i, h in enumerate(histograms):
        try:
            results.append(fit(h))
        except FitError as e:
            failed.append((i, str(e)))
    return results, failed
```

</details>

## 9. Summary

* Polymorphism: code written for an interface works for every object that keeps it.
* Inherit for "is a", one or two levels deep; compose for "has a".
* An abstract base class documents and enforces an interface; duck typing does not need it.
* Exceptions: name the class you expect, keep the `try` short, raise early.

<details>
<summary><b>[i] Cheat sheet — Inheritance and exceptions</b></summary>

| | |
|:--|:--|
| `class B(A):` | `B` inherits from `A`: attributes and methods |
| override | redefine an inherited method in `B` |
| `super().__init__(...)` | call the base class version |
| `class A(ABC)` + `@abstractmethod` | an interface: cannot be instantiated, derived classes must implement |
| `C.__mro__` | the order in which methods are looked up |
| duck typing | Python only needs the method to exist |
| "is a" / "has a" | inheritance / composition: **prefer composition** |
| `try / except E / else / finally` | first matching `except`; `else` if no error; `finally` always |
| `class MyError(ValueError): ...` | your own exception, under the closest built-in |

**Traps**: inherited methods that create objects of the base class; a derived class
that cannot keep the promises of its base; bare `except:`.

</details>

### Check yourself

1. Why does `simulate` need no change when a new force model arrives?
2. What does an abstract method force, and when is the error raised?
3. Why does `total_area` work with `Ring`, which does not derive from `Shape`?
4. Give an example of "is a" and one of "has a" from a physics code.
5. `except ArithmeticError` catches a `ZeroDivisionError`. Why?
6. Why is `except:` without a class a bad idea?

<details>
<summary><b>Solutions</b></summary>

1. It only calls `model.force(x, v)`: any object with that method works (polymorphism).
2. Every concrete derived class must implement it; otherwise creating an object raises a
   `TypeError`.
3. Duck typing: `total_area` only calls `area()`, and `Ring` has it.
4. A `Calorimeter` is a `Detector`; an `Event` has `Track`s.
5. `ZeroDivisionError` derives from `ArithmeticError`: an `except` catches the class and
   all its derived classes.
6. It catches everything, also the errors you did not expect (typos, `Ctrl-C`), and
   hides them.

</details>